# HouseValue-ML: California Housing Price Prediction

Interactive walkthrough matching `train.py` for **ITE 351: AI & Applications (Stage A)**.

1. Exploratory Data Analysis (EDA)
2. Leak-free Preprocessing & 80/20 Split
3. Multi-Model Training & 5-Fold Cross-Validation
4. Quantitative Evaluation & Permutation Feature Importance
5. Single-Sample Inference

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')

## 1. Dataset Loading & Exploratory Data Analysis

In [2]:
dataset = fetch_california_housing(as_frame=True)
df = dataset.frame.copy()
print('Shape:', df.shape)
print('Missing values:', int(df.isna().sum().sum()))
capped = (df['MedHouseVal'] >= 5.0).sum()
print(f'Capped >= 5.0 ($500k): {capped} ({capped / len(df) * 100:.2f}%)')
df.describe().T[['mean', 'std', 'min', '50%', 'max']].round(4)

Shape: (20640, 9)
Missing values: 0
Capped >= 5.0 ($500k): 992 (4.81%)


,mean,std,min,50%,max
MedInc,3.8707,1.8998,0.4999,3.5348,15.0001
HouseAge,28.6395,12.5856,1.0000,29.0000,52.0000
AveRooms,5.4290,2.4742,0.8462,5.2291,141.9091
AveBedrms,1.0967,0.4739,0.3333,1.0488,34.0667
Population,1425.4767,1132.4621,3.0000,1166.0000,35682.0000
AveOccup,3.0707,10.3860,0.6923,2.8181,1243.3333
Latitude,35.6319,2.1360,32.5400,34.2600,41.9500
Longitude,-119.5697,2.0035,-124.3500,-118.4900,-114.3100
MedHouseVal,2.0686,1.1540,0.1500,1.7970,5.0000


In [3]:
df.corr(numeric_only=True)['MedHouseVal'].drop('MedHouseVal').sort_values(ascending=False).round(4)

MedInc        0.6881
AveRooms      0.1519
HouseAge      0.1056
AveOccup     -0.0237
Population   -0.0246
Longitude    -0.0460
AveBedrms    -0.0467
Latitude     -0.1442
Name: MedHouseVal, dtype: float64

## 2. Preprocessing & 80/20 Train/Test Split

In [4]:
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

Train: 16512 | Test: 4128


## 3. Model Training, 5-Fold Cross-Validation & Test Evaluation

In [5]:
models = {
    'Linear Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearRegression()),
    ]),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ),
    'HistGradientBoosting': HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.08, max_leaf_nodes=31, random_state=RANDOM_STATE
    ),
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
fitted = {}
for name, est in models.items():
    est.fit(X_train, y_train)
    preds = est.predict(X_test)
    rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
    mae = float(mean_absolute_error(y_test, preds))
    r2 = float(r2_score(y_test, preds))
    cv_res = cross_validate(est, X_train, y_train, cv=cv, scoring='r2', n_jobs=-1)
    rows.append({
        'Model': name,
        'Test MAE': round(mae, 4),
        'Test RMSE': round(rmse, 4),
        'Test R2': round(r2, 4),
        '5-Fold CV R2 Mean': round(float(cv_res['test_score'].mean()), 4),
    })
    fitted[name] = est

pd.DataFrame(rows)

,Model,Test MAE,Test RMSE,Test R2,5-Fold CV R2 Mean
0,Linear Regression,0.5332,0.7456,0.5758,0.6115
1,Random Forest,0.3266,0.5039,0.8062,0.8058
2,HistGradientBoosting,0.2945,0.4457,0.8484,0.8400


## 4. Permutation Feature Importance (Best Model)

In [6]:
best_model = fitted['HistGradientBoosting']
perm = permutation_importance(
    best_model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1, scoring='r2'
)
pd.DataFrame({
    'Feature': X_test.columns,
    'Importance Mean': perm.importances_mean.round(4),
    'Importance Std': perm.importances_std.round(4),
}).sort_values('Importance Mean', ascending=False)

,Feature,Importance Mean,Importance Std
6,Latitude,0.9875,0.0207
7,Longitude,0.8346,0.0115
0,MedInc,0.4145,0.0114
5,AveOccup,0.1469,0.0044
2,AveRooms,0.0676,0.0042
1,HouseAge,0.0431,0.0024
4,Population,0.0087,0.0006
3,AveBedrms,0.0060,0.0013


## 5. Single-Sample Inference

In [7]:
def predict_house_value(model, sample_features: dict) -> float:
    cols = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
    sample_df = pd.DataFrame([[sample_features[c] for c in cols]], columns=cols)
    return round(float(model.predict(sample_df)[0]) * 100_000.0, 2)

coastal_bay = {
    'MedInc': 7.80, 'HouseAge': 32.0, 'AveRooms': 6.40, 'AveBedrms': 1.08,
    'Population': 1150.0, 'AveOccup': 2.60, 'Latitude': 37.75, 'Longitude': -122.42,
}
inland_valley = {
    'MedInc': 3.10, 'HouseAge': 22.0, 'AveRooms': 5.10, 'AveBedrms': 1.05,
    'Population': 1600.0, 'AveOccup': 3.10, 'Latitude': 36.75, 'Longitude': -119.77,
}
print(f'Coastal Bay Area District: ${predict_house_value(best_model, coastal_bay):,.2f}')
print(f'Inland Central Valley District: ${predict_house_value(best_model, inland_valley):,.2f}')

Coastal Bay Area District: $441,008.02
Inland Central Valley District: $72,920.19
